# 05 · Latent selectors (Table 3, rows A5–A9)

Selection rules for the SAE latents that need no k-means pseudo-groups — the entropy of a latent's activations
(A5), its dip statistic (A6), co-firing with other latents (A7, all pairs; A8, exclusive pairs) — and an oracle
that scores the latents by MI with the true classes (A9), each at N = 1–32 latents with Directions-LEACE, on Table 3's
two settings, seeds 0–4. Run after notebook 04 (the random-latent draws are checked against its band runs).
Results: `results/selectors/` (the per-split SAEs are cached in `results/selectors/sae/`).

The published runs fitted the SAEs on a CPU, which the table's build checks; set `FIT_DEVICE = None` to use a GPU
instead (the SAEs, and so the numbers, then differ slightly). About 40 minutes on a laptop CPU.

In [ ]:
# On Colab: mount Google Drive and move to your copy of the repository (see the README: "Running on Google Colab";
# set REPO if your copy is elsewhere). Skipped outside Colab.
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/label-free-deconfounding-embeddings'
    %cd {REPO}
    %pip install -q -r requirements.txt
except ImportError:
    print('not on Colab: using the current working directory')

In [ ]:
import os, sys
if not os.path.isdir('code') and os.path.isdir(os.path.join('..', 'code')):
    os.chdir('..')                                   # opened from notebooks/
assert os.path.isdir('code') and os.path.isdir('data'), f'run from the repository root; cwd={os.getcwd()}'
sys.path.insert(0, 'code')

import torch                                         # first: some modules load diptest, whose macOS wheel bundles libomp
import paths

# QUICK = True runs every section end to end in minutes on a subset (two seeds, a 20-epoch SAE, a few settings),
# writing to results/*_quick/ so it never mixes with the real results. Its numbers are NOT the paper's.
QUICK = False
paths.configure(quick_run=QUICK)                     # before the experiment modules are imported
DEVICE = 'cuda' if torch.cuda.is_available() else None   # None: each step's own choice
print('cwd:', os.getcwd(), '| device:', DEVICE or 'auto', '| QUICK' if QUICK else '')

In [ ]:
import artifacts, experiments as E
import latent_selectors as LS

artifacts.set_results_dir(paths.SELECTORS)
E.set_embed_cache(paths.EMBED_CACHE)
CELLS = (('swiltra_de_fr_it', 'e5-small'), ('scotus_oyez_lexis_wiki', 'e5-small'))
FIT_DEVICE = 'cpu'
RUN_KW = dict(seeds=LS.SEEDS, budgets=LS.BUDGETS, selectors=LS.SELECTORS, draws=LS.DRAWS, gauss_draws=LS.GAUSS_DRAWS,
              nonlinear=('mlp',), nonlinear_budgets=LS.DETAIL, fit_device=FIT_DEVICE)
if QUICK:
    CELLS = CELLS[:1]
    RUN_KW.update(seeds=(0,), budgets=(2, 8), draws=2, nonlinear=(), nonlinear_budgets=(), n_epochs=20)
print('selectors', LS.SELECTORS, '| budgets', RUN_KW['budgets'], '| seeds', RUN_KW['seeds'])

In [ ]:
LS.run_cells(CELLS, **RUN_KW)